# 02 — Expanded Relative Valuation

This notebook moves from data preparation to valuation analysis. It uses the processed GMAT3 dataset from notebook 01 and expands the peer group beyond the original Brazilian listed peers.

The goal is to estimate an implied valuation range for Grupo Mateus (`GMAT3.SA`) using trading comparables and scenario-based EV/EBITDA multiples. This is not an investment recommendation and should not be read as a single definitive target price.

The analysis now separates three ideas: domestic listed peers, operational peers that may not be tradable, and broader LatAm food retail peers. That separation matters because peer comparability is a judgment call, not just a spreadsheet exercise.


In [13]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:,.2f}".format)


## 1. Load Inputs

The notebook starts with three inputs.

First, `master_valuation_dataset.csv` comes from notebook 01 and contains GMAT3, Assai, and GPA with market data and calculated EV/EBITDA. Second, `peer_universe.csv` defines the strategic peer universe. Third, `external_peer_multiples.csv` adds current EV/EBITDA multiples for selected LatAm peers.

The external peer multiples are used as market multiple references. Their enterprise value and EBITDA are in local currencies, but EV/EBITDA itself is currency-neutral because both numerator and denominator use the same currency.


In [14]:
processed_data_path = Path("../data/processed/master_valuation_dataset.csv")
peer_universe_path = Path("../data/raw/peer_universe.csv")
external_peer_multiples_path = Path("../data/raw/external_peer_multiples.csv")
source_log_path = Path("../data/raw/source_log.csv")

valuation_df = pd.read_csv(processed_data_path)
peer_universe_df = pd.read_csv(peer_universe_path)
external_peer_multiples_df = pd.read_csv(external_peer_multiples_path)
source_log_df = pd.read_csv(source_log_path)

peer_universe_df["include_in_trading_comps"] = (
    peer_universe_df["include_in_trading_comps"].astype(str).str.lower().eq("true")
)

display(valuation_df)
display(peer_universe_df)
display(external_peer_multiples_df)
display(source_log_df)


,ticker,company,sector,subsector,period,report_date,currency,revenue,ebitda,ebit,net_income,cash,total_debt,shares_outstanding,valuation_date,current_price,market_cap,market_cap_mn,net_debt,enterprise_value,ebitda_annualized,ev_ebitda_annualized
0,GMAT3.SA,Grupo Mateus,Consumer Staples,Food Retail,1T26,2026-03-31,BRL,9402,400,NaN,213,1984,2720,"2,300,047,621.00",2026-05-28,4.26,9798203392,"9,798.20",736,"10,534.20",1600,6.58
1,ASAI3.SA,Assai,Consumer Staples,Cash & Carry,1T26,2026-03-31,BRL,20600,1000,NaN,174,4366,16374,"1,353,531,000.00",2026-05-28,9.25,12411186176,"12,411.19",12008,"24,419.19",4000,6.10
2,PCAR3.SA,GPA,Consumer Staples,Food Retail,1T26,2026-03-31,BRL,4374,458,NaN,-1347,943,4173,NaN,2026-05-28,1.99,978954240,978.95,3230,"4,208.95",1832,2.30


,ticker,company,country,exchange,currency,subsector,peer_role,peer_quality,include_in_trading_comps,notes
0,GMAT3.SA,Grupo Mateus,Brazil,B3,BRL,Food Retail and Cash & Carry,Target,Target,False,Target company to be valued
1,ASAI3.SA,Assai,Brazil,B3,BRL,Cash & Carry,Domestic Core Peer,Core Peer,True,Closest listed Brazilian cash and carry compar...
2,PCAR3.SA,GPA,Brazil,B3,BRL,Food Retail,Domestic Distressed Peer,Distressed / Turnaround Peer,True,Listed Brazilian food retail peer but distress...
3,CRFB3.SA,Carrefour Brasil / Atacadao,Brazil,B3 delisted,BRL,Food Retail and Cash & Carry,Domestic Operational Peer,Delisted / Operational Reference,False,Operationally relevant but not a current tradi...
4,WALMEX.MX,Walmart de Mexico,Mexico,BMV,MXN,Food Retail and Clubs,LatAm Peer,High Quality LatAm Peer,True,Large LatAm food retail and club-store operator
5,CHDRAUIB.MX,Grupo Comercial Chedraui,Mexico,BMV,MXN,Food Retail,LatAm Peer,LatAm Food Retail Peer,True,Mexican food retail operator with Mexico and U...
6,SORIANAB.MX,Organizacion Soriana,Mexico,BMV,MXN,Food Retail,LatAm Peer,LatAm Food Retail Peer,True,Mexican supermarket and food retail operator
7,CENCOSUD.SN,Cencosud,Chile,Santiago,CLP,Food Retail and Multi-format Retail,LatAm Peer,LatAm Multi-format Retail Peer,True,Large Chilean and regional retail operator wit...


,as_of_date,ticker,company,country,exchange,currency,current_price,market_cap,enterprise_value,ebitda,ev_ebitda,multiple_source,notes
0,2026-05-30,WALMEX.MX,Walmart de Mexico,Mexico,BMV,MXN,52.40,906111942656,959297290240,97478295552,9.84,yfinance,Enterprise value and EBITDA in local currency;...
1,2026-05-30,CHDRAUIB.MX,Grupo Comercial Chedraui,Mexico,BMV,MXN,97.43,93508042752,141098713088,19550976000,7.22,yfinance,Enterprise value and EBITDA in local currency;...
2,2026-05-30,SORIANAB.MX,Organizacion Soriana,Mexico,BMV,MXN,34.49,61025402880,81359486976,10997213184,7.40,yfinance,Enterprise value and EBITDA in local currency;...
3,2026-05-30,CENCOSUD.SN,Cencosud,Chile,Santiago,CLP,"2,099.00",5780666843136,11398445793280,1049218449408,10.86,yfinance,Enterprise value and EBITDA in local currency;...


## 2. Peer Universe Logic

GMAT3 is the target company. Assai remains the cleanest domestic listed peer. GPA is still relevant to Brazilian food retail, but it is a distressed / turnaround peer and should be treated with caution.

Carrefour Brasil / Atacadao is operationally relevant, especially because of the cash-and-carry overlap, but it is no longer a current listed trading comparable after its delisting. It remains in the peer universe as context, not as a live market multiple.

The four additional peers are LatAm listed food retail references: Walmex, Chedraui, Soriana, and Cencosud. They improve sample size and regional comparability, but they also introduce differences in geography, scale, business mix, currency, and market structure.


In [15]:
target_ticker = "GMAT3.SA"

target_df = valuation_df[valuation_df["ticker"].eq(target_ticker)].copy()

peer_universe_display_columns = [
    "ticker",
    "company",
    "country",
    "exchange",
    "subsector",
    "peer_role",
    "peer_quality",
    "include_in_trading_comps",
    "notes",
]

peer_universe_df[peer_universe_display_columns]


,ticker,company,country,exchange,subsector,peer_role,peer_quality,include_in_trading_comps,notes
0,GMAT3.SA,Grupo Mateus,Brazil,B3,Food Retail and Cash & Carry,Target,Target,False,Target company to be valued
1,ASAI3.SA,Assai,Brazil,B3,Cash & Carry,Domestic Core Peer,Core Peer,True,Closest listed Brazilian cash and carry compar...
2,PCAR3.SA,GPA,Brazil,B3,Food Retail,Domestic Distressed Peer,Distressed / Turnaround Peer,True,Listed Brazilian food retail peer but distress...
3,CRFB3.SA,Carrefour Brasil / Atacadao,Brazil,B3 delisted,Food Retail and Cash & Carry,Domestic Operational Peer,Delisted / Operational Reference,False,Operationally relevant but not a current tradi...
4,WALMEX.MX,Walmart de Mexico,Mexico,BMV,Food Retail and Clubs,LatAm Peer,High Quality LatAm Peer,True,Large LatAm food retail and club-store operator
5,CHDRAUIB.MX,Grupo Comercial Chedraui,Mexico,BMV,Food Retail,LatAm Peer,LatAm Food Retail Peer,True,Mexican food retail operator with Mexico and U...
6,SORIANAB.MX,Organizacion Soriana,Mexico,BMV,Food Retail,LatAm Peer,LatAm Food Retail Peer,True,Mexican supermarket and food retail operator
7,CENCOSUD.SN,Cencosud,Chile,Santiago,Food Retail and Multi-format Retail,LatAm Peer,LatAm Multi-format Retail Peer,True,Large Chilean and regional retail operator wit...


## 3. Build Expanded Multiple Table

Notebook 01 already calculated EV/EBITDA for GMAT3, Assai, and GPA using BRL financials and market data. For the expanded LatAm peers, this notebook imports external EV/EBITDA observations from the raw multiple snapshot.

This creates one comparable table with the same analytical field: `ev_ebitda_multiple`. Carrefour Brasil is visible in the peer universe, but it is not included in trading multiple statistics because there is no current CRFB3 market quote after delisting.


In [16]:
local_multiple_df = valuation_df[
    [
        "ticker",
        "company",
        "valuation_date",
        "current_price",
        "market_cap_mn",
        "enterprise_value",
        "ebitda_annualized",
        "net_income",
        "ev_ebitda_annualized",
    ]
].copy()

local_multiple_df = local_multiple_df.rename(
    columns={
        "valuation_date": "as_of_date",
        "ev_ebitda_annualized": "ev_ebitda_multiple",
    }
)
local_multiple_df["multiple_source"] = "notebook_01_annualized_1q26"
local_multiple_df["reported_currency"] = "BRL"
local_multiple_df["ebitda_basis"] = "1Q26 annualized"
local_multiple_df["net_income_basis"] = "1Q26 annualized"
local_multiple_df["net_income_annualized"] = local_multiple_df["net_income"] * 4
local_multiple_df["pe_ratio"] = np.where(
    local_multiple_df["net_income_annualized"] > 0,
    local_multiple_df["market_cap_mn"] / local_multiple_df["net_income_annualized"],
    np.nan,
)
local_multiple_df["pe_status"] = np.where(
    local_multiple_df["net_income_annualized"] > 0,
    "Calculated",
    "N.M. - negative earnings",
)

external_multiple_df = external_peer_multiples_df[
    [
        "as_of_date",
        "ticker",
        "company",
        "currency",
        "current_price",
        "market_cap",
        "enterprise_value",
        "ebitda",
        "net_income",
        "ev_ebitda",
        "pe_ratio",
        "multiple_source",
    ]
].copy()

external_multiple_df = external_multiple_df.rename(
    columns={
        "currency": "reported_currency",
        "market_cap": "market_cap_mn",
        "ebitda": "ebitda_annualized",
        "net_income": "net_income_annualized",
        "ev_ebitda": "ev_ebitda_multiple",
    }
)
external_multiple_df["ebitda_basis"] = "yfinance trailing EBITDA"
external_multiple_df["net_income_basis"] = "yfinance trailing net income"
external_multiple_df["pe_status"] = np.where(
    external_multiple_df["pe_ratio"].notna() & (external_multiple_df["pe_ratio"] > 0),
    "Calculated",
    "N.M.",
)

expanded_peer_multiples_df = pd.concat(
    [local_multiple_df, external_multiple_df],
    ignore_index=True,
    sort=False,
)

expanded_peer_multiples_df = expanded_peer_multiples_df.merge(
    peer_universe_df[
        [
            "ticker",
            "country",
            "exchange",
            "subsector",
            "peer_role",
            "peer_quality",
            "include_in_trading_comps",
            "notes",
        ]
    ],
    on="ticker",
    how="left",
)

normalized_valuation_base_df = expanded_peer_multiples_df[
    [
        "ticker",
        "company",
        "country",
        "peer_role",
        "peer_quality",
        "include_in_trading_comps",
        "reported_currency",
        "ebitda_basis",
        "net_income_basis",
        "ev_ebitda_multiple",
        "pe_ratio",
        "pe_status",
        "multiple_source",
    ]
].copy()

normalized_valuation_base_df


,ticker,company,country,peer_role,peer_quality,include_in_trading_comps,reported_currency,ebitda_basis,ev_ebitda_multiple,multiple_source
0,GMAT3.SA,Grupo Mateus,Brazil,Target,Target,False,BRL,1Q26 annualized,6.58,notebook_01_annualized_1q26
1,ASAI3.SA,Assai,Brazil,Domestic Core Peer,Core Peer,True,BRL,1Q26 annualized,6.10,notebook_01_annualized_1q26
2,PCAR3.SA,GPA,Brazil,Domestic Distressed Peer,Distressed / Turnaround Peer,True,BRL,1Q26 annualized,2.30,notebook_01_annualized_1q26
3,WALMEX.MX,Walmart de Mexico,Mexico,LatAm Peer,High Quality LatAm Peer,True,MXN,yfinance trailing EBITDA,9.84,yfinance
4,CHDRAUIB.MX,Grupo Comercial Chedraui,Mexico,LatAm Peer,LatAm Food Retail Peer,True,MXN,yfinance trailing EBITDA,7.22,yfinance
5,SORIANAB.MX,Organizacion Soriana,Mexico,LatAm Peer,LatAm Food Retail Peer,True,MXN,yfinance trailing EBITDA,7.40,yfinance
6,CENCOSUD.SN,Cencosud,Chile,LatAm Peer,LatAm Multi-format Retail Peer,True,CLP,yfinance trailing EBITDA,10.86,yfinance


## 4. Trading Comparable Set

The trading comparable set excludes GMAT3 because the target should not be used to value itself. It also excludes operational references that are not currently tradable, such as Carrefour Brasil after delisting.

GPA remains in the table, but its distressed profile must be isolated in scenario design. A peer can be visible and still receive a lower analytical weight.


In [17]:
trading_peers_df = expanded_peer_multiples_df[
    expanded_peer_multiples_df["include_in_trading_comps"]
    & expanded_peer_multiples_df["ticker"].ne(target_ticker)
    & expanded_peer_multiples_df["ev_ebitda_multiple"].notna()
].copy()

trading_peers_df = trading_peers_df.sort_values("ev_ebitda_multiple")

trading_peers_df[
    [
        "ticker",
        "company",
        "country",
        "peer_quality",
        "ev_ebitda_multiple",
        "notes",
    ]
]


,ticker,company,country,peer_quality,ev_ebitda_multiple,notes
2,PCAR3.SA,GPA,Brazil,Distressed / Turnaround Peer,2.30,Listed Brazilian food retail peer but distress...
1,ASAI3.SA,Assai,Brazil,Core Peer,6.10,Closest listed Brazilian cash and carry compar...
4,CHDRAUIB.MX,Grupo Comercial Chedraui,Mexico,LatAm Food Retail Peer,7.22,Mexican food retail operator with Mexico and U...
5,SORIANAB.MX,Organizacion Soriana,Mexico,LatAm Food Retail Peer,7.40,Mexican supermarket and food retail operator
3,WALMEX.MX,Walmart de Mexico,Mexico,High Quality LatAm Peer,9.84,Large LatAm food retail and club-store operator
6,CENCOSUD.SN,Cencosud,Chile,LatAm Multi-format Retail Peer,10.86,Large Chilean and regional retail operator wit...


## 5. Peer Multiple Statistics

The core multiple remains EV/EBITDA. Since the peer group is still small and heterogeneous, this notebook calculates several statistical cuts instead of relying on one headline number.

The full peer set includes GPA and therefore captures a distressed lower bound. The ex-distressed set removes GPA and focuses on healthier listed food retail references.


In [18]:
full_trading_multiple_series = trading_peers_df["ev_ebitda_multiple"].dropna()
ex_distressed_peers_df = trading_peers_df[
    ~trading_peers_df["peer_quality"].str.contains("Distressed", case=False, na=False)
].copy()
ex_distressed_multiple_series = ex_distressed_peers_df["ev_ebitda_multiple"].dropna()

peer_stats_df = pd.DataFrame(
    [
        {
            "peer_set": "Full Trading Peer Set",
            "count": full_trading_multiple_series.count(),
            "mean": full_trading_multiple_series.mean(),
            "median": full_trading_multiple_series.median(),
            "p25": full_trading_multiple_series.quantile(0.25),
            "p75": full_trading_multiple_series.quantile(0.75),
            "min": full_trading_multiple_series.min(),
            "max": full_trading_multiple_series.max(),
        },
        {
            "peer_set": "Ex-Distressed Trading Peer Set",
            "count": ex_distressed_multiple_series.count(),
            "mean": ex_distressed_multiple_series.mean(),
            "median": ex_distressed_multiple_series.median(),
            "p25": ex_distressed_multiple_series.quantile(0.25),
            "p75": ex_distressed_multiple_series.quantile(0.75),
            "min": ex_distressed_multiple_series.min(),
            "max": ex_distressed_multiple_series.max(),
        },
    ]
)

peer_stats_df


,peer_set,count,mean,median,p25,p75,min,max
0,Full Trading Peer Set,6,7.29,7.31,6.38,9.23,2.30,10.86
1,Ex-Distressed Trading Peer Set,5,8.28,7.40,7.22,9.84,6.10,10.86


## 6. Scenario Design

The valuation scenarios are designed to answer different questions.

The Domestic Core Case asks what GMAT3 would be worth using Assai only. The Domestic Listed Median includes GPA and shows how much a distressed peer can pull the benchmark down. The Expanded LatAm ex-Distressed Median uses the broader regional peer group while excluding GPA. The Expanded LatAm including GPA Median shows the mechanical impact of including every traded peer.

The Conservative Case uses the 25th percentile of the full traded set. The Upside Case uses the higher of GMAT3's current multiple and the 75th percentile of the ex-distressed peer set. This keeps the range analytical without pretending that one exact multiple is the answer.


## 5A. P/L Multiples

P/L, or Price-to-Earnings, is calculated as Market Cap divided by net income. For the Brazilian companies, net income is annualized from 1Q26 as a practical proxy. For LatAm peers, the notebook uses the yfinance trailing net income snapshot.

GPA has negative earnings in the current base, so its P/L is classified as `N.M.` (not meaningful). This is not a technical error; it is an analytical conclusion that P/L should not be used as a normal valuation anchor for GPA.


In [ ]:
pe_comparison_df = expanded_peer_multiples_df[
    [
        "ticker",
        "company",
        "country",
        "peer_quality",
        "market_cap_mn",
        "net_income_annualized",
        "net_income_basis",
        "pe_ratio",
        "pe_status",
    ]
].copy()

pe_comparison_df


## 5B. Sector Net Income

Sector net income needs careful treatment because the expanded peer group uses different currencies. It would be incorrect to sum Brazilian BRL earnings with Mexican MXN earnings and Chilean CLP earnings.

For this reason, the notebook calculates Brazilian sector net income separately in BRL millions and uses P/L multiples for cross-country comparison. GPA is shown both included and excluded, because its negative earnings materially distort the domestic sector profit pool.


In [ ]:
brazil_net_income_df = expanded_peer_multiples_df[
    expanded_peer_multiples_df["country"].eq("Brazil")
    & expanded_peer_multiples_df["net_income_annualized"].notna()
].copy()

sector_net_income_records = [
    {
        "sector_cut": "Brazil listed food retail incl. GMAT3 and GPA",
        "currency": "BRL",
        "net_income_basis": "1Q26 annualized",
        "companies": ", ".join(brazil_net_income_df["ticker"]),
        "company_count": len(brazil_net_income_df),
        "sector_net_income": brazil_net_income_df["net_income_annualized"].sum(),
        "positive_earners_net_income": brazil_net_income_df.loc[
            brazil_net_income_df["net_income_annualized"] > 0, "net_income_annualized"
        ].sum(),
    },
    {
        "sector_cut": "Brazil peers only incl. GPA",
        "currency": "BRL",
        "net_income_basis": "1Q26 annualized",
        "companies": ", ".join(
            brazil_net_income_df.loc[brazil_net_income_df["ticker"].ne(target_ticker), "ticker"]
        ),
        "company_count": len(brazil_net_income_df.loc[brazil_net_income_df["ticker"].ne(target_ticker)]),
        "sector_net_income": brazil_net_income_df.loc[
            brazil_net_income_df["ticker"].ne(target_ticker), "net_income_annualized"
        ].sum(),
        "positive_earners_net_income": brazil_net_income_df.loc[
            brazil_net_income_df["ticker"].ne(target_ticker)
            & (brazil_net_income_df["net_income_annualized"] > 0),
            "net_income_annualized",
        ].sum(),
    },
    {
        "sector_cut": "Brazil peers ex-distressed GPA",
        "currency": "BRL",
        "net_income_basis": "1Q26 annualized",
        "companies": ", ".join(
            brazil_net_income_df.loc[
                brazil_net_income_df["ticker"].isin(["ASAI3.SA"]), "ticker"
            ]
        ),
        "company_count": len(brazil_net_income_df.loc[brazil_net_income_df["ticker"].isin(["ASAI3.SA"])]),
        "sector_net_income": brazil_net_income_df.loc[
            brazil_net_income_df["ticker"].isin(["ASAI3.SA"]), "net_income_annualized"
        ].sum(),
        "positive_earners_net_income": brazil_net_income_df.loc[
            brazil_net_income_df["ticker"].isin(["ASAI3.SA"]), "net_income_annualized"
        ].sum(),
    },
]

sector_net_income_df = pd.DataFrame(sector_net_income_records)
sector_net_income_df


In [19]:
core_brazil_multiple = trading_peers_df.loc[
    trading_peers_df["ticker"].eq("ASAI3.SA"), "ev_ebitda_multiple"
].iloc[0]

domestic_listed_median = trading_peers_df.loc[
    trading_peers_df["ticker"].isin(["ASAI3.SA", "PCAR3.SA"]),
    "ev_ebitda_multiple",
].median()

expanded_latam_ex_distressed_median = ex_distressed_multiple_series.median()
expanded_latam_including_gpa_median = full_trading_multiple_series.median()
conservative_multiple = full_trading_multiple_series.quantile(0.25)

target_current_ev_ebitda = target_df["ev_ebitda_annualized"].iloc[0]
ex_distressed_p75 = ex_distressed_multiple_series.quantile(0.75)

if pd.notna(target_current_ev_ebitda):
    upside_multiple = max(target_current_ev_ebitda, ex_distressed_p75)
else:
    upside_multiple = ex_distressed_p75

scenario_records = [
    {
        "scenario": "Conservative Case",
        "applied_ev_ebitda_multiple": conservative_multiple,
        "peer_count": len(trading_peers_df),
        "included_peers": ", ".join(trading_peers_df["ticker"]),
        "methodology": "25th percentile of full traded peer set, including GPA",
    },
    {
        "scenario": "Domestic Listed Median",
        "applied_ev_ebitda_multiple": domestic_listed_median,
        "peer_count": 2,
        "included_peers": "ASAI3.SA, PCAR3.SA",
        "methodology": "Median of Brazilian listed peers, including distressed GPA",
    },
    {
        "scenario": "Domestic Core Case",
        "applied_ev_ebitda_multiple": core_brazil_multiple,
        "peer_count": 1,
        "included_peers": "ASAI3.SA",
        "methodology": "Assai-only case as closest domestic listed comparable",
    },
    {
        "scenario": "Expanded LatAm ex-Distressed Median",
        "applied_ev_ebitda_multiple": expanded_latam_ex_distressed_median,
        "peer_count": len(ex_distressed_peers_df),
        "included_peers": ", ".join(ex_distressed_peers_df["ticker"]),
        "methodology": "Median of traded peers excluding distressed GPA and delisted Carrefour Brasil",
    },
    {
        "scenario": "Expanded LatAm including GPA Median",
        "applied_ev_ebitda_multiple": expanded_latam_including_gpa_median,
        "peer_count": len(trading_peers_df),
        "included_peers": ", ".join(trading_peers_df["ticker"]),
        "methodology": "Median of all traded peers, including GPA",
    },
    {
        "scenario": "Upside Case",
        "applied_ev_ebitda_multiple": upside_multiple,
        "peer_count": len(ex_distressed_peers_df),
        "included_peers": ", ".join(ex_distressed_peers_df["ticker"]),
        "methodology": "Higher of GMAT3 current EV/EBITDA and ex-distressed peer 75th percentile",
    },
]

scenario_multiples_df = pd.DataFrame(scenario_records)
scenario_multiples_df


,scenario,applied_ev_ebitda_multiple,peer_count,included_peers,methodology
0,Conservative Case,6.38,6,"PCAR3.SA, ASAI3.SA, CHDRAUIB.MX, SORIANAB.MX, ...","25th percentile of full traded peer set, inclu..."
1,Domestic Listed Median,4.20,2,"ASAI3.SA, PCAR3.SA","Median of Brazilian listed peers, including di..."
2,Domestic Core Case,6.10,1,ASAI3.SA,Assai-only case as closest domestic listed com...
3,Expanded LatAm ex-Distressed Median,7.40,5,"ASAI3.SA, CHDRAUIB.MX, SORIANAB.MX, WALMEX.MX,...",Median of traded peers excluding distressed GP...
4,Expanded LatAm including GPA Median,7.31,6,"PCAR3.SA, ASAI3.SA, CHDRAUIB.MX, SORIANAB.MX, ...","Median of all traded peers, including GPA"
5,Upside Case,9.84,5,"ASAI3.SA, CHDRAUIB.MX, SORIANAB.MX, WALMEX.MX,...",Higher of GMAT3 current EV/EBITDA and ex-distr...


## 7. Implied Enterprise Value

The applied multiple is multiplied by GMAT3 annualized EBITDA to estimate implied enterprise value.

Formula:

`Implied EV = GMAT3 EBITDA annualized × applied EV/EBITDA multiple`

GMAT3 EBITDA is still based on 1Q26 annualized EBITDA from notebook 01. That is a practical first-pass approximation, not a substitute for LTM EBITDA.


In [20]:
target_ebitda_annualized = target_df["ebitda_annualized"].iloc[0]
target_net_debt = target_df["net_debt"].iloc[0]
target_shares_outstanding = target_df["shares_outstanding"].iloc[0]
target_current_price = target_df["current_price"].iloc[0]
target_current_market_cap_mn = target_df["market_cap_mn"].iloc[0]
target_current_ev = target_df["enterprise_value"].iloc[0]
target_current_ev_ebitda = target_df["ev_ebitda_annualized"].iloc[0]

valuation_scenarios_df = scenario_multiples_df.copy()
valuation_scenarios_df["target_ebitda_annualized"] = target_ebitda_annualized
valuation_scenarios_df["implied_enterprise_value"] = (
    valuation_scenarios_df["target_ebitda_annualized"]
    * valuation_scenarios_df["applied_ev_ebitda_multiple"]
)

valuation_scenarios_df[
    [
        "scenario",
        "applied_ev_ebitda_multiple",
        "target_ebitda_annualized",
        "implied_enterprise_value",
        "methodology",
    ]
]


,scenario,applied_ev_ebitda_multiple,target_ebitda_annualized,implied_enterprise_value,methodology
0,Conservative Case,6.38,1600,"10,212.56","25th percentile of full traded peer set, inclu..."
1,Domestic Listed Median,4.20,1600,"6,721.81","Median of Brazilian listed peers, including di..."
2,Domestic Core Case,6.10,1600,"9,767.67",Assai-only case as closest domestic listed com...
3,Expanded LatAm ex-Distressed Median,7.40,1600,"11,836.80",Median of traded peers excluding distressed GP...
4,Expanded LatAm including GPA Median,7.31,1600,"11,692.00","Median of all traded peers, including GPA"
5,Upside Case,9.84,1600,"15,745.60",Higher of GMAT3 current EV/EBITDA and ex-distr...


## 8. Equity Value Bridge

Enterprise Value represents the value of the whole operating business. To estimate equity value, subtract net debt.

Formula:

`Equity Value = Enterprise Value - Net Debt`

Then:

`Implied Price per Share = Equity Value / Shares Outstanding`

Because EV and net debt are in BRL millions while shares are absolute, equity value is multiplied by 1,000,000 before dividing by shares.


In [21]:
valuation_scenarios_df["target_net_debt"] = target_net_debt
valuation_scenarios_df["implied_equity_value"] = (
    valuation_scenarios_df["implied_enterprise_value"]
    - valuation_scenarios_df["target_net_debt"]
)

valuation_scenarios_df["implied_price_per_share"] = (
    valuation_scenarios_df["implied_equity_value"] * 1_000_000
) / target_shares_outstanding

valuation_scenarios_df["upside_downside_pct"] = np.where(
    pd.notna(target_current_price) & (target_current_price != 0),
    (valuation_scenarios_df["implied_price_per_share"] / target_current_price - 1) * 100,
    np.nan,
)

valuation_scenarios_df[
    [
        "scenario",
        "applied_ev_ebitda_multiple",
        "implied_enterprise_value",
        "implied_equity_value",
        "implied_price_per_share",
        "upside_downside_pct",
        "peer_count",
    ]
]


,scenario,applied_ev_ebitda_multiple,implied_enterprise_value,implied_equity_value,implied_price_per_share,upside_downside_pct,peer_count
0,Conservative Case,6.38,"10,212.56","9,476.56",4.12,-3.28,6
1,Domestic Listed Median,4.20,"6,721.81","5,985.81",2.60,-38.91,2
2,Domestic Core Case,6.10,"9,767.67","9,031.67",3.93,-7.82,1
3,Expanded LatAm ex-Distressed Median,7.40,"11,836.80","11,100.80",4.83,13.29,5
4,Expanded LatAm including GPA Median,7.31,"11,692.00","10,956.00",4.76,11.82,6
5,Upside Case,9.84,"15,745.60","15,009.60",6.53,53.19,5


## 9. Current Market Positioning

This table compares GMAT3's current EV/EBITDA with the expanded peer set. The objective is to understand where GMAT3 is already priced relative to domestic and LatAm benchmarks.

A lower multiple is not automatically cheaper, and a higher multiple is not automatically expensive. Differences in growth, leverage, profitability, country risk, liquidity, and accounting basis can justify valuation gaps.


In [22]:
current_positioning_df = expanded_peer_multiples_df[
    expanded_peer_multiples_df["ticker"].isin(
        [target_ticker] + trading_peers_df["ticker"].tolist()
    )
].copy()

current_positioning_df = current_positioning_df[
    [
        "ticker",
        "company",
        "country",
        "peer_quality",
        "ev_ebitda_multiple",
        "pe_ratio",
        "pe_status",
        "include_in_trading_comps",
        "multiple_source",
    ]
].sort_values("ev_ebitda_multiple")

current_positioning_df


,ticker,company,country,peer_quality,ev_ebitda_multiple,include_in_trading_comps,multiple_source
2,PCAR3.SA,GPA,Brazil,Distressed / Turnaround Peer,2.30,True,notebook_01_annualized_1q26
1,ASAI3.SA,Assai,Brazil,Core Peer,6.10,True,notebook_01_annualized_1q26
0,GMAT3.SA,Grupo Mateus,Brazil,Target,6.58,False,notebook_01_annualized_1q26
4,CHDRAUIB.MX,Grupo Comercial Chedraui,Mexico,LatAm Food Retail Peer,7.22,True,yfinance
5,SORIANAB.MX,Organizacion Soriana,Mexico,LatAm Food Retail Peer,7.40,True,yfinance
3,WALMEX.MX,Walmart de Mexico,Mexico,High Quality LatAm Peer,9.84,True,yfinance
6,CENCOSUD.SN,Cencosud,Chile,LatAm Multi-format Retail Peer,10.86,True,yfinance


## 10. Formatted Valuation Table

This table is for reading and presentation only. It does not replace the numeric `valuation_scenarios_df`, which should remain the analytical source for exports, charts, and any later model extensions.


In [23]:
display_df = valuation_scenarios_df[
    [
        "scenario",
        "applied_ev_ebitda_multiple",
        "implied_enterprise_value",
        "implied_equity_value",
        "implied_price_per_share",
        "upside_downside_pct",
        "peer_count",
    ]
].copy()

display_df["applied_ev_ebitda_multiple"] = display_df[
    "applied_ev_ebitda_multiple"
].apply(lambda value: "-" if pd.isna(value) else f"{value:,.1f}x")

display_df["implied_enterprise_value"] = display_df[
    "implied_enterprise_value"
].apply(lambda value: "-" if pd.isna(value) else f"R$ {value:,.0f} mi")

display_df["implied_equity_value"] = display_df[
    "implied_equity_value"
].apply(lambda value: "-" if pd.isna(value) else f"R$ {value:,.0f} mi")

display_df["implied_price_per_share"] = display_df[
    "implied_price_per_share"
].apply(lambda value: "-" if pd.isna(value) else f"R$ {value:,.2f}")

display_df["upside_downside_pct"] = display_df[
    "upside_downside_pct"
].apply(lambda value: "-" if pd.isna(value) else f"{value:,.1f}%")

display_df


,scenario,applied_ev_ebitda_multiple,implied_enterprise_value,implied_equity_value,implied_price_per_share,upside_downside_pct,peer_count
0,Conservative Case,6.4x,"R$ 10,213 mi","R$ 9,477 mi",R$ 4.12,-3.3%,6
1,Domestic Listed Median,4.2x,"R$ 6,722 mi","R$ 5,986 mi",R$ 2.60,-38.9%,2
2,Domestic Core Case,6.1x,"R$ 9,768 mi","R$ 9,032 mi",R$ 3.93,-7.8%,1
3,Expanded LatAm ex-Distressed Median,7.4x,"R$ 11,837 mi","R$ 11,101 mi",R$ 4.83,13.3%,5
4,Expanded LatAm including GPA Median,7.3x,"R$ 11,692 mi","R$ 10,956 mi",R$ 4.76,11.8%,6
5,Upside Case,9.8x,"R$ 15,746 mi","R$ 15,010 mi",R$ 6.53,53.2%,5


## 11. Operating Thesis and Retail Drivers

A multiple-based valuation is only credible if the operating story explains why the company should trade at a discount, in line, or at a premium to peers.

For GMAT3, the constructive thesis relies on five retail drivers. First, the company has a strong regional position, especially in markets where local execution and supplier relationships matter. Second, store maturation can support growth even before considering new openings. Third, logistics and distribution are central to food retail competitiveness, particularly in regions where supply chain density can create cost advantages. Fourth, EBITDA margin should be compared with peers because the market pays higher multiples for retailers that combine growth with resilient profitability. Fifth, competition in cash-and-carry remains the main operational risk, especially against Assai, Atacadao/Carrefour and regional players.

This is why the valuation conclusion should be constructive but not aggressive: GMAT3 looks moderately discounted versus a healthier LatAm peer set, but the upside depends on execution, margins and store maturity.


## 12. Initial Interpretation

The expanded peer set changes the valuation conversation. GMAT3 currently trades close to Assai on EV/EBITDA, which supports the idea that the domestic market already prices GMAT3 broadly in line with the cleanest Brazilian listed comparable.

GPA must be treated explicitly as a stress case, not as the main benchmark for fair value. Its distressed / turnaround profile and negative earnings make P/L not meaningful and make its multiples less representative of a normalized food retail operator. GPA can remain in the analysis, but mainly to show downside and stress, not to define the central valuation range.

Carrefour Brasil / Atacadao is operationally relevant, but after delisting it is no longer a live trading comparable. It should help the qualitative sector discussion, not the market multiple median.

The LatAm peers expand the sample and reduce dependence on a one-peer Assai case. However, they introduce new differences in country risk, scale, profitability, business mix, FX exposure, and market liquidity. The expanded ex-distressed median is therefore useful as a triangulation point, not as a final target price.

The base conclusion remains constructive but not aggressive: GMAT3 does not look absurdly cheap versus Assai, but it appears moderately discounted when compared with a healthier LatAm peer set excluding GPA as a distressed distortion.


In [24]:
outputs_dir = Path("../outputs")
outputs_dir.mkdir(parents=True, exist_ok=True)

valuation_scenarios_path = outputs_dir / "gmat3_relative_valuation_scenarios.csv"
current_positioning_path = outputs_dir / "current_market_positioning.csv"
expanded_peer_multiples_path = outputs_dir / "expanded_peer_multiples.csv"
peer_stats_path = outputs_dir / "expanded_peer_stats.csv"
pe_comparison_path = outputs_dir / "pe_comparison.csv"
sector_net_income_path = outputs_dir / "sector_net_income.csv"
normalized_base_path = Path("../data/processed/normalized_valuation_base.csv")
source_log_output_path = outputs_dir / "source_log.csv"

valuation_scenarios_df.to_csv(valuation_scenarios_path, index=False)
current_positioning_df.to_csv(current_positioning_path, index=False)
expanded_peer_multiples_df.to_csv(expanded_peer_multiples_path, index=False)
peer_stats_df.to_csv(peer_stats_path, index=False)
pe_comparison_df.to_csv(pe_comparison_path, index=False)
sector_net_income_df.to_csv(sector_net_income_path, index=False)
normalized_valuation_base_df.to_csv(normalized_base_path, index=False)
source_log_df.to_csv(source_log_output_path, index=False)

print(f"Valuation scenarios exported to: {valuation_scenarios_path}")
print(f"Current positioning exported to: {current_positioning_path}")
print(f"Expanded peer multiples exported to: {expanded_peer_multiples_path}")
print(f"Peer statistics exported to: {peer_stats_path}")
print(f"P/E comparison exported to: {pe_comparison_path}")
print(f"Sector net income exported to: {sector_net_income_path}")
print(f"Normalized valuation base exported to: {normalized_base_path}")
print(f"Source log exported to: {source_log_output_path}")


Valuation scenarios exported to: ../outputs/gmat3_relative_valuation_scenarios.csv
Current positioning exported to: ../outputs/current_market_positioning.csv
Expanded peer multiples exported to: ../outputs/expanded_peer_multiples.csv
Peer statistics exported to: ../outputs/expanded_peer_stats.csv


## 12. Next Steps

The next notebook should turn the valuation range into a more complete research exhibit. It should include visualizations, sensitivity tables, a football field chart, and a structured peer inclusion/exclusion discussion.

The analysis should also evaluate whether US peers such as Costco, Kroger, BJ's, and Walmart are useful as secondary global references or too structurally different for the main valuation set.

Eventually, the project should produce a final research summary that separates mechanical valuation output from analyst judgment, clearly documents assumptions, and avoids turning a first-pass multiple range into an investment recommendation.
